# Module 2 — Topic 4: Transforming Data
## Live Demo Notebook

**Scenario:** The startup's CEO has asked for a business performance report before the quarterly review. She wants answers to five specific questions:

1. Which states are our top revenue generators?
2. How does each product category perform — orders, revenue, and average order value?
3. What is our revenue breakdown by payment method for delivered orders only?
4. Which individual orders are our top 10 highest-value transactions?
5. How does performance compare across market regions (North vs South)?

We'll answer all five using `orders_clean.csv` — the clean dataset from Topic 3.

---

## Part 1 — Load the Clean Dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../../data/orders_clean.csv", encoding="utf-8-sig",
                 parse_dates=["order_date"])

print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")
print("Columns:", df.columns.tolist())
df.head(3)

---
## Part 2 — Filtering: Pulling the Right Rows

In [ ]:
# Delivered orders only — the only orders where revenue is confirmed
delivered = df[df["delivery_status"] == "Delivered"]
print(f"Delivered orders: {len(delivered)} of {len(df)} total ({len(delivered)/len(df)*100:.1f}%)")

In [ ]:
# High-value Electronics orders — for premium product analysis
premium_electronics = df[
    (df["product_category"] == "Electronics") &
    (df["total_amount"] > 20000)
]
print(f"Premium Electronics orders: {len(premium_electronics)}")
premium_electronics[["order_id", "customer_name", "state", "product_name", "total_amount"]].head(6)

In [ ]:
# Northern Nigerian states — regional analysis
northern = df[df["state"].isin(["Kano", "Kaduna", "Abuja"])]
southern = df[~df["state"].isin(["Kano", "Kaduna", "Abuja"])]

print(f"Northern states orders: {len(northern)}")
print(f"Southern states orders: {len(southern)}")
print()
print(f"Northern revenue: NGN {northern['total_amount'].sum():,.0f}")
print(f"Southern revenue: NGN {southern['total_amount'].sum():,.0f}")

---
## Part 3 — Sorting: Finding Top and Bottom Performers

In [ ]:
# Question 4: Top 10 highest-value orders
top10_orders = (
    df.nlargest(10, "total_amount")
      .reset_index(drop=True)
      [["order_id", "customer_name", "state", "product_category",
        "product_name", "quantity", "total_amount"]]
)

print("===== Top 10 Orders by Value =====")
top10_orders

In [ ]:
# Sort by state (A→Z) then by revenue within each state (highest first)
by_state_value = (
    df.sort_values(["state", "total_amount"], ascending=[True, False])
      .reset_index(drop=True)
)

print("Top orders per state (first 12 rows):")
by_state_value[["order_id", "state", "product_category", "total_amount"]].head(12)

---
## Part 4 — Adding Computed Columns

In [ ]:
# Revenue per item ordered
df["revenue_per_item"] = (df["total_amount"] / df["quantity"]).round(2)

# VAT-inclusive amount (Nigerian VAT = 7.5%)
df["amount_with_vat"] = (df["total_amount"] * 1.075).round(2)

# Location string
df["location"] = df["city"] + ", " + df["state"]

print("New columns added. Sample:")
df[["order_id", "quantity", "total_amount", "revenue_per_item",
    "amount_with_vat", "location"]].head(6)

In [ ]:
# Extract month name and quarter from order_date
df["order_month_name"] = df["order_date"].dt.strftime("%B")
df["order_quarter"]    = df["order_date"].dt.quarter
df["day_of_week"]      = df["order_date"].dt.day_name()

print("Orders per quarter:")
print(df["order_quarter"].value_counts().sort_index())
print()
print("Orders per day of week:")
print(df["day_of_week"].value_counts())

---
## Part 5 — `apply()`: Custom Classification Logic

In [ ]:
# Classify orders into business tiers
def assign_tier(amount):
    if amount > 30000:
        return "Premium"
    elif amount > 10000:
        return "Mid-range"
    else:
        return "Standard"

df["order_tier"] = df["total_amount"].apply(assign_tier)

print("Order tier distribution:")
tier_counts = df["order_tier"].value_counts()
for tier, count in tier_counts.items():
    pct = count / len(df) * 100
    print(f"  {tier:<12}: {count:>3} orders ({pct:.1f}%)")

In [ ]:
# apply() with axis=1 — uses multiple columns per row
def market_priority(row):
    """Flag high-value orders in Tier 1 markets for priority fulfilment."""
    tier1_states = ["Lagos", "Abuja", "Rivers"]
    if row["total_amount"] > 20000 and row["state"] in tier1_states:
        return "Priority"
    elif row["total_amount"] > 10000:
        return "Standard"
    else:
        return "Regular"

df["fulfilment_priority"] = df.apply(market_priority, axis=1)

print("Fulfilment priority distribution:")
print(df["fulfilment_priority"].value_counts())

---
## Part 6 — `groupby()`: Answering the Business Questions

In [ ]:
# Question 1: Which states are our top revenue generators?
state_revenue = (
    df.groupby("state")["total_amount"]
      .agg(orders="count", total_revenue="sum", avg_order="mean")
      .round(0)
      .sort_values("total_revenue", ascending=False)
)

print("===== Revenue by State =====")
state_revenue

In [ ]:
# Question 2: How does each product category perform?
category_summary = (
    df.groupby("product_category")["total_amount"]
      .agg(
          order_count="count",
          total_revenue="sum",
          avg_order="mean",
          max_order="max",
          min_order="min"
      )
      .round(0)
      .sort_values("total_revenue", ascending=False)
)

print("===== Category Performance Summary =====")
category_summary

In [ ]:
# Question 3: Revenue breakdown by payment method — delivered orders only
payment_summary = (
    df[df["delivery_status"] == "Delivered"]
      .groupby("payment_method")["total_amount"]
      .agg(orders="count", revenue="sum", avg_order="mean")
      .round(0)
      .sort_values("revenue", ascending=False)
)

print("===== Payment Method Performance (Delivered Orders Only) =====")
payment_summary

In [ ]:
# Group by TWO columns — state × category
state_category = (
    df.groupby(["state", "product_category"])["total_amount"]
      .sum()
      .round(0)
      .reset_index()
      .rename(columns={"total_amount": "revenue"})
      .sort_values("revenue", ascending=False)
)

print("Top 10 state × category combinations by revenue:")
state_category.head(10)

---
## Part 7 — `transform()`: Group Stats on Individual Rows

In [ ]:
# Add each order's share of its category's total revenue
df["category_revenue_total"] = (
    df.groupby("product_category")["total_amount"].transform("sum")
)

df["category_share_pct"] = (
    df["total_amount"] / df["category_revenue_total"] * 100
).round(3)

# Show the most impactful individual orders
top_shares = (
    df.nlargest(8, "category_share_pct")
      [["order_id", "product_category", "total_amount",
        "category_revenue_total", "category_share_pct"]]
)

print("Orders with the largest share of their category's revenue:")
top_shares

---
## Part 8 — Merging: Adding Regional Context

In [ ]:
# Regional lookup table — market tier classification
region_lookup = pd.DataFrame({
    "state": ["Lagos", "Abuja", "Kano", "Rivers", "Oyo",
              "Kaduna", "Enugu", "Delta", "Anambra", "Ogun"],
    "region": ["South-West", "North-Central", "North-West", "South-South", "South-West",
               "North-West", "South-East", "South-South", "South-East", "South-West"],
    "market_tier": ["Tier 1", "Tier 1", "Tier 2", "Tier 1", "Tier 2",
                   "Tier 2", "Tier 2", "Tier 2", "Tier 2", "Tier 2"],
})

print("Region lookup table:")
region_lookup

In [ ]:
# Left join — keep all orders, add region and market_tier
df_merged = df.merge(region_lookup, on="state", how="left")

print(f"Shape before merge: {df.shape}")
print(f"Shape after merge:  {df_merged.shape}")
print()
df_merged[["order_id", "state", "region", "market_tier", "total_amount"]].head(8)

In [ ]:
# Question 5: Revenue comparison by market region and tier
region_summary = (
    df_merged.groupby(["region", "market_tier"])["total_amount"]
      .agg(orders="count", revenue="sum", avg_order="mean")
      .round(0)
      .sort_values("revenue", ascending=False)
)

print("===== Revenue by Region & Market Tier =====")
region_summary

---
## Part 9 — The Complete CEO Report

Answering all five questions in one chained pipeline each.

In [ ]:
print("=" * 55)
print("        QUARTERLY PERFORMANCE REPORT")
print("=" * 55)

total_revenue  = df["total_amount"].sum()
total_orders   = len(df)
delivered_pct  = (df["delivery_status"] == "Delivered").mean() * 100
avg_order      = df["total_amount"].mean()

print(f"\nTotal orders:       {total_orders}")
print(f"Total revenue:      NGN {total_revenue:,.0f}")
print(f"Average order:      NGN {avg_order:,.0f}")
print(f"Delivery rate:      {delivered_pct:.1f}%")

print("\n--- Top 3 States by Revenue ---")
top_states = df.groupby("state")["total_amount"].sum().nlargest(3)
for state, rev in top_states.items():
    print(f"  {state:<12}: NGN {rev:>10,.0f}")

print("\n--- Category Performance ---")
cat_perf = df.groupby("product_category")["total_amount"].agg(
    orders="count", revenue="sum"
).sort_values("revenue", ascending=False)
for cat, row in cat_perf.iterrows():
    share = row["revenue"] / total_revenue * 100
    print(f"  {cat:<20}: {int(row['orders']):>3} orders | "
          f"NGN {row['revenue']:>10,.0f} ({share:.1f}%)")

print("\n--- Top Payment Method (Delivered Orders) ---")
top_payment = (
    df[df["delivery_status"] == "Delivered"]
      .groupby("payment_method")["total_amount"]
      .sum()
      .idxmax()
)
print(f"  {top_payment}")

print("\n--- Top 3 Individual Orders ---")
top3 = df.nlargest(3, "total_amount")[["order_id", "customer_name",
                                        "product_category", "total_amount"]]
for _, row in top3.iterrows():
    print(f"  {row['order_id']} | {row['customer_name']:<20} | "
          f"{row['product_category']:<20} | NGN {row['total_amount']:>10,.0f}")

print("\n" + "=" * 55)

---
## Summary

In this demo we:
- Loaded `orders_clean.csv` — the output from Topic 3
- Filtered rows using single conditions, multiple conditions, `.isin()`, and `~` (NOT)
- Sorted by single and multiple columns, with `reset_index(drop=True)`
- Added computed columns: arithmetic, string concat, datetime extraction
- Used `apply()` with a named function and `axis=1` for multi-column logic
- Answered three business questions with `groupby().agg()`
- Grouped by two columns simultaneously (`state` × `product_category`)
- Used `transform()` to broadcast category totals back to individual rows
- Merged with a regional lookup table using `how='left'`
- Produced a complete formatted business report in Part 9

**Next — Topic 5:** Descriptive Statistics — understanding distributions, measuring spread, detecting outliers, and exploring relationships between variables.